### Requirements

In [1]:
%%capture

# Core training libraries
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.1 \
    evaluate==0.4.2

# Utilities
!pip install -q \
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm



# Evaluation (requires pydantic v2)
!pip install -q --upgrade pydantic
!pip install -q google-genai rouge-score
!pip install --upgrade bitsandbytes transformers accelerate
!pip install -U trl

print("✅ Installation complete!")
print("✅ All dependencies compatible (pydantic v2 + google-genai)")

In [2]:
pip install numpy==1.26.4 pandas==2.2.2 pyarrow==15.0.2 datasets

  Using cached pyarrow-15.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.8.2-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.8.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.8.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.7.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.6.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.6.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.4.0-py3-none-any

In [3]:
import sys
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: CUDA not available. Training will be VERY slow on CPU.")

print("="*60)

ENVIRONMENT CHECK
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
Device name: Tesla T4
Device capability: (7, 5)
Total VRAM: 14.56 GB


### HuggingFace Login

In [4]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_token')
!hf auth login --token $HF_TOKEN

print("ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Bootcamp_FineTuning` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.


### Confings

In [5]:
import torch
from pprint import pprint

# Auto-detect compute dtype (BF16 requires compute capability >= 8.0)
use_bf16 = False
compute_dtype = torch.float16

CONFIG = {
    # Model
    "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    # Alternative for tighter VRAM: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # For GGUF export, prefer: "meta-llama/Llama-3.2-3B-Instruct" or Mistral models

    # Dataset
    "dataset_name": "/content/train.jsonl",
    "dataset_split": "train",
    "dataset_subsample": 500,  # Colab-safe: 500 | Local: 1500
    "train_val_split": 0.9,  # 90% train, 10% validation

    # Tokenization
    "max_length": 256,  # Colab: 512 | Local: 1024

    # Training
    "num_train_epochs": 1,
    "max_steps": 150,  # Colab: 250 | Local: 600
    "per_device_train_batch_size": 1,  # Colab: 1 | Local: 2
    "gradient_accumulation_steps": 16,  # Colab: 64 | Local: 32
    "learning_rate": 2e-5,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 50,
    "eval_steps": 100,
    "save_total_limit": 2,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 128,
    "temperature": 0.0,  # Deterministic
    "do_sample": True,

    # HF credentials
    'hf_username': 'Piyumi Imasha',
    'hub_model_name': 'ledger-mindt',
}

# Effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION (COLAB FREE TIER)")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)

CONFIGURATION (COLAB FREE TIER)
{'base_model': 'mistralai/Mistral-7B-Instruct-v0.2',
 'bnb_4bit_compute_dtype': torch.float16,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_use_double_quant': True,
 'dataset_name': '/content/train.jsonl',
 'dataset_split': 'train',
 'dataset_subsample': 500,
 'do_sample': True,
 'eval_steps': 100,
 'gradient_accumulation_steps': 16,
 'hf_username': 'Piyumi Imasha',
 'hub_model_name': 'ledger-mindt',
 'learning_rate': 2e-05,
 'load_in_4bit': True,
 'logging_steps': 10,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'lora_r': 16,
 'lora_target_modules': ['q_proj',
                         'k_proj',
                         'v_proj',
                         'o_proj',
                         'gate_proj',
                         'up_proj',
                         'down_proj'],
 'max_length': 256,
 'max_new_tokens': 128,
 'max_steps': 150,
 'num_train_epochs': 1,
 'output_dir': 'outputs/adapter',
 'per_device_train_batch_size': 1,
 'push_to_hub': False,
 'save_s

### Load Data

In [6]:
from datasets import load_dataset, Dataset
import json

SEED = 42

dataset = load_dataset(
    "json",
    data_files=CONFIG["dataset_name"],
    split=CONFIG["dataset_split"]
)

# Apply subsampling if specified
if CONFIG["dataset_subsample"] and len(dataset) > CONFIG["dataset_subsample"]:
    dataset = dataset.shuffle(seed=SEED).select(range(CONFIG["dataset_subsample"]))

print(f"Loaded dataset with {len(dataset)} examples.")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Loaded dataset with 500 examples.
Dataset({
    features: ['question', 'answer', 'chunk_index', 'source', 'qa_pair_id'],
    num_rows: 500
})


### Train, val Split

In [7]:
# Remove unwanted columns
dataset = dataset.map(
    lambda x: x,
    remove_columns=['chunk_index', 'source', 'qa_pair_id']
)

# Split into train/validation
split_dataset = dataset.train_test_split(
    train_size=CONFIG["train_val_split"],
    seed=SEED
)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"📊 Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("\n📝 Sample example:")
print(train_dataset[0])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

📊 Train: 450 | Validation: 50

📝 Sample example:
{'question': "How may litigation or other claims arising from acquisitions impact the acquirer's liabilities?", 'answer': 'Information not available in the provided context.'}


In [8]:
import pandas as pd

# Convert first 50 samples to dataframe
df_preview = pd.DataFrame(train_dataset[:50])

# Display with formatting
pd.set_option('display.max_colwidth', 100)  # Limit column width for readability
print(f"📊 Displaying first 50 samples out of {len(dataset)} total examples\n")
df_preview

📊 Displaying first 50 samples out of 500 total examples



,question,answer
0,How may litigation or other claims arising from acquisitions impact the acquirer's liabilities?,Information not available in the provided context.
1,What is the duration of the measurement period for adjustments to assets and liabilities acquire...,The duration of the measurement period for adjustments to assets and liabilities acquired during...
2,What is the exclusive forum for resolving complaints asserting a cause of action arising under t...,The federal district courts of the United States of America.
3,"According to the document, what costs are primarily included in sales and marketing expenses?","Sales and marketing expenses primarily consist of advertising costs, product marketing costs, co..."
4,"How might required changes to the qualification, screening, and background check process affect ...",The time required to recruit new Drivers could extend.
5,When was the constitutional reform to the judiciary in Mexico approved?,A constitutional reform to the judiciary in Mexico was approved in September 2024.
6,"According to the document, in which areas are legislative bodies and regulatory entities conside...",Legislative bodies and regulatory entities are considering proposals related to the company's bu...
7,"What is the purpose of classifying Drivers as employees, workers, or quasi-employees?","The purpose of classifying Drivers as employees, workers, or quasi-employees is not explicitly s..."
8,"According to the document, why would remediation efforts be necessary if an audit determines def...",Remediation efforts may distract our management team and require their time and resources.
9,What type of losses could the company incur due to the negative publicity and unfavorable media ...,The company could incur losses due to negative publicity and unfavorable media coverage resultin...


In [9]:
from transformers import AutoTokenizer
import numpy as np

# Load tokenizer for diagnostics
print(f"Loading tokenizer: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Sample up to 500 examples for diagnostics
sample_size = min(500, len(train_dataset))
sample_dataset = train_dataset.select(range(sample_size))

# Compute token lengths
token_lengths = []
for example in sample_dataset:
    # Concatenate instruction + input + output
    text = f"{example['question']} {example['answer']}"
    tokens = tokenizer(text, add_special_tokens=True)
    token_lengths.append(len(tokens["input_ids"]))

token_lengths = np.array(token_lengths)

print("="*60)
print("TOKEN LENGTH DIAGNOSTICS")
print("="*60)
print(f"Sample size: {sample_size}")
print(f"Average token length: {token_lengths.mean():.1f}")
print(f"Median token length: {np.median(token_lengths):.1f}")
print(f"Min token length: {token_lengths.min()}")
print(f"Max token length: {token_lengths.max()}")
print(f"95th percentile: {np.percentile(token_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(token_lengths, 99):.1f}")
print()
truncated = (token_lengths > CONFIG["max_length"]).sum()
truncation_rate = truncated / len(token_lengths) * 100
print(f"Truncation at max_length={CONFIG['max_length']}: {truncated}/{len(token_lengths)} ({truncation_rate:.1f}%)")
print("="*60)

Loading tokenizer: mistralai/Mistral-7B-Instruct-v0.2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

TOKEN LENGTH DIAGNOSTICS
Sample size: 450
Average token length: 48.8
Median token length: 47.0
Min token length: 20
Max token length: 105
95th percentile: 76.0
99th percentile: 93.5

Truncation at max_length=256: 0/450 (0.0%)


In [10]:
def chatml_format(user_text, system_text="You are an internal Uber assistant.", assistant_text=None):
    """Format text in ChatML style."""
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    if assistant_text:
        messages.append({"role": "assistant", "content": assistant_text})

    # Format as ChatML
    formatted = ""
    for msg in messages:
        formatted += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"

    return formatted


def build_sft_prompt(row):
    """Build SFT prompt from dataset row."""
    user_text = row["question"]

    prompt = chatml_format(
        user_text=user_text,
        system_text="You are an internal Uber assistant.",
        assistant_text=row["answer"]
    )

    return {"text": prompt}


# Map datasets to text format
train_dataset = train_dataset.map(build_sft_prompt)
val_dataset = val_dataset.map(build_sft_prompt)

print("✅ Prompts built for SFT")
print("\n📝 Sample formatted prompt:")
print(train_dataset[0]["text"][:500] + "...")

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Prompts built for SFT

📝 Sample formatted prompt:
<|im_start|>system
You are an internal Uber assistant.<|im_end|>
<|im_start|>user
How may litigation or other claims arising from acquisitions impact the acquirer's liabilities?<|im_end|>
<|im_start|>assistant
Information not available in the provided context.<|im_end|>
...


### 4-bit quantization

In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import time
import torch

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["load_in_4bit"],
    bnb_4bit_compute_dtype=CONFIG["bnb_4bit_compute_dtype"],
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=CONFIG["bnb_4bit_use_double_quant"],
)

print(f"📥 Loading base model: {CONFIG['base_model']}...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("✅ Base model loaded in 4-bit")

# Test prompts
test_prompts = [
    {
        "title": "Uber Strategy Alignment",
        "prompt": "You are an internal Uber assistant. What are Uber’s key strategic priorities for 2024, and how should an intern align their project work with these goals? Answer in a concise, professional, and data-driven tone."
    },
    {
        "title": "Driver Earnings Feature Guidance",
        "prompt": "You are an Uber internal assistant helping interns. I'm building a feature to improve driver earnings. What factors should I consider to ensure it aligns with Uber’s business goals and maintains marketplace balance?"
    },
]

print("\n" + "="*60)
print("BASELINE OUTPUTS (PRE-FINETUNE)")
print("="*60)

if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM before generation: {vram_before:.2f} GB\n")

for i, test in enumerate(test_prompts, 1):
    # Format as ChatML
    formatted_prompt = chatml_format(test["prompt"])

    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(base_model.device)

    # Generate
    start_time = time.time()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"] if CONFIG["temperature"] > 0 else None,
            do_sample=CONFIG["do_sample"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the assistant response
    if "<|im_start|>assistant" in generated_text:
        response = generated_text.split("<|im_start|>assistant")[-1].split("<|im_end|>")[0].strip()
    else:
        response = generated_text[len(formatted_prompt):].strip()

    tokens_generated = outputs.shape[1] - inputs["input_ids"].shape[1]
    tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0

    print(f"\n[{i}] {test['title']}")
    print("-" * 60)
    print(f"Prompt: {test['prompt'][:100]}...")
    print(f"\nResponse:\n{response}")
    print(f"\nLatency: {elapsed:.2f}s | Tokens: {tokens_generated} | Speed: {tokens_per_sec:.1f} tok/s")
    print("-" * 60)

if torch.cuda.is_available():
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"\nVRAM after generation: {vram_after:.2f} GB")
    print(f"VRAM delta: {vram_after - vram_before:.2f} GB")

print("\n" + "="*60)

📥 Loading base model: mistralai/Mistral-7B-Instruct-v0.2...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Base model loaded in 4-bit

BASELINE OUTPUTS (PRE-FINETUNE)
VRAM before generation: 3.84 GB


[1] Uber Strategy Alignment
------------------------------------------------------------
Prompt: You are an internal Uber assistant. What are Uber’s key strategic priorities for 2024, and how shoul...

Response:
As an assistant, I don't have the ability to access Uber's most current or confidential information. However, based on Uber's previous statements and investor expectations, I can provide some insights into potential strategic priorities for 2024 and how an intern might align their project work with these goals.
1. Expanding Uber's mobility offerings: Uber is likely to continue investing in new mobility services beyond ride-hailing, such as micromobility, electric bikes, and autonomous vehicles. An intern could contribute to this goal by researching emerging mobility technologies and trends

Latency: 13.48s | Tokens: 128 | Speed: 9.5 tok/s
---------------------------------------------

### LORA Configuration

In [12]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

# LoRA Configuration - targeting ONLY attention projection layers as specified
lora_config = LoraConfig(
    r=CONFIG["lora_r"],                    # Rank
    lora_alpha=CONFIG["lora_alpha"],       # Scaling factor
    lora_dropout=CONFIG["lora_dropout"],   # Dropout
    target_modules=CONFIG["lora_target_modules"],  # q_proj, k_proj, v_proj, o_proj ONLY
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA adapters
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
    print(f"Total params: {all_params:,}")

print("="*60)
print("LoRA CONFIGURATION")
print("="*60)
print(f"Rank (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Dropout: {lora_config.lora_dropout}")
print(f"Target modules: {lora_config.target_modules}")
print()
print_trainable_parameters(model)
print("="*60)

LoRA CONFIGURATION
Rank (r): 16
Alpha: 32
Dropout: 0.05
Target modules: {'up_proj', 'q_proj', 'v_proj', 'gate_proj', 'down_proj', 'k_proj', 'o_proj'}

Trainable params: 41,943,040 (1.11%)
Total params: 3,794,014,208


In [13]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256, # Reduced from 512 to save VRAM
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
print("Dataset re-tokenized with max_length=256 to optimize VRAM usage.")

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Dataset re-tokenized with max_length=256 to optimize VRAM usage.


### SFT Trainer Configuration

In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Clear CUDA cache before initialization
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ CUDA cache cleared.")

# CONSTRAINT: Ensure minimum 100 training steps
MIN_STEPS = 100
max_steps = max(CONFIG["max_steps"], MIN_STEPS)

print(f"Enforcing minimum {MIN_STEPS} training steps. Actual max_steps: {max_steps}")

# Training Arguments optimized for Tesla T4 (FP16)
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    max_steps=max_steps,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"], # Set to 64 for effective batch size 64
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_total_limit=CONFIG["save_total_limit"],
    fp16=False, # T4 compatibility
    bf16=False, # T4 does not support BF16 for this op
    optim="adamw_torch",
    report_to="none",
    load_best_model_at_end=False,
    gradient_checkpointing=True, # VRAM optimization
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("="*60)
print("UPDATED TRAINING CONFIGURATION (T4 OPTIMIZED)")
print("="*60)
print(f"Max steps: {max_steps}")
print(f"Effective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"FP16 enabled: {training_args.fp16}")
print(f"Gradient Checkpointing: {training_args.gradient_checkpointing}")
print("="*60)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ CUDA cache cleared.
Enforcing minimum 100 training steps. Actual max_steps: 150


Truncating train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

UPDATED TRAINING CONFIGURATION (T4 OPTIMIZED)
Max steps: 150
Effective batch size: 16
FP16 enabled: False
Gradient Checkpointing: True


In [15]:
# Removed manual FP32 conversion to prevent OutOfMemoryError
# for param in model.parameters():
#     param.data = param.data.float()

print("Skipping manual float conversion to save VRAM.")

# Re-verify trainable parameters after LoRA application
print_trainable_parameters(model)

# Clear cache to ensure maximum free memory for training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Current VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Skipping manual float conversion to save VRAM.
Trainable params: 41,943,040 (1.11%)
Total params: 3,794,014,208
Current VRAM usage: 4.42 GB


In [16]:
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

### Training

In [17]:
import torch

# Final check and cache clearing to avoid OOM
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Initial VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

print("="*60)
print("STARTING TRAINING")
print("="*60)

try:
    # Start the fine-tuning process
    train_result = trainer.train()

    # Print training metrics
    print("\n" + "="*60)
    print("TRAINING COMPLETED")
    print("="*60)
    print(f"Training loss: {train_result.training_loss:.4f}")
    print(f"Total steps: {train_result.global_step}")

    if torch.cuda.is_available():
        vram_after_train = torch.cuda.memory_allocated() / 1024**3
        print(f"VRAM after training: {vram_after_train:.2f} GB")
        print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

except Exception as e:
    print(f"Training failed with error: {e}")
    if "out of memory" in str(e).lower():
        print("\nTIP: Try further reducing per_device_train_batch_size or max_length.")

print("="*60)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Initial VRAM usage: 4.42 GB
STARTING TRAINING


Step,Training Loss,Validation Loss
100,0.383577,0.394651



TRAINING COMPLETED
Training loss: 0.6637
Total steps: 150
VRAM after training: 4.58 GB
Peak VRAM: 12.39 GB


### Save Adapter

In [18]:
import os

# Save LoRA adapters
adapter_path = os.path.join(CONFIG["output_dir"], "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("="*60)
print("ADAPTERS SAVED")
print("="*60)
print(f"Adapter path: {adapter_path}")
print(f"Files saved:")
for f in os.listdir(adapter_path):
    file_size = os.path.getsize(os.path.join(adapter_path, f)) / 1024**2
    print(f"  - {f} ({file_size:.2f} MB)")
print("="*60)

# Optionally push to HuggingFace Hub
if CONFIG["push_to_hub"]:
    hub_name = f"{CONFIG['hf_username']}/{CONFIG['hub_model_name']}"
    print(f"\nPushing to HuggingFace Hub: {hub_name}")
    model.push_to_hub(hub_name)
    tokenizer.push_to_hub(hub_name)
    print(f"Successfully pushed to {hub_name}")

ADAPTERS SAVED
Adapter path: outputs/adapter/final_adapter
Files saved:
  - tokenizer_config.json (0.00 MB)
  - adapter_model.safetensors (80.06 MB)
  - README.md (0.00 MB)
  - chat_template.jinja (0.00 MB)
  - adapter_config.json (0.00 MB)
  - tokenizer.json (3.34 MB)


### Inference Pipeline

In [20]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

def load_finetuned_model(base_model_name, adapter_path, load_in_4bit=True):
    """Load the base model with LoRA adapters for inference."""

    # Quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=load_in_4bit,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Load LoRA adapters
    model = PeftModel.from_pretrained(base_model, adapter_path)

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def query_intern(question, model=None, tokenizer=None, max_new_tokens=128, temperature=0.0):
    """
    Query the fine-tuned intern assistant model.

    Args:
        question (str): The question to ask the model
        model: The loaded model (uses global inference_model if None)
        tokenizer: The tokenizer (uses global inference_tokenizer if None)
        max_new_tokens (int): Maximum tokens to generate
        temperature (float): Sampling temperature (0.0 for deterministic)

    Returns:
        str: The model's response

    Example:
        >>> response = query_intern("What are Uber's key strategic priorities?")
        >>> print(response)
    """
    # Use global model/tokenizer if not provided
    if model is None:
        model = globals().get('inference_model', None)
    if tokenizer is None:
        tokenizer = globals().get('inference_tokenizer', None)

    if model is None or tokenizer is None:
        raise ValueError("Model and tokenizer must be loaded first. Call load_finetuned_model().")

    # Format prompt using ChatML format
    prompt = f"""<|im_start|>system
You are an internal Uber assistant.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if temperature > 0 else None,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract assistant response
    if "<|im_start|>assistant" in generated_text:
        response = generated_text.split("<|im_start|>assistant")[-1]
        response = response.split("<|im_end|>")[0].strip()
    else:
        # Fallback: remove input tokens
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    return response


print("="*60)
print("INFERENCE PIPELINE FUNCTIONS DEFINED")
print("="*60)
print("  - load_finetuned_model(base_model_name, adapter_path)")
print("  - query_intern(question, model, tokenizer)")
print("="*60)

INFERENCE PIPELINE FUNCTIONS DEFINED
  - load_finetuned_model(base_model_name, adapter_path)
  - query_intern(question, model, tokenizer)


In [21]:
# Load the fine-tuned model for inference
# Option 1: Use the model already in memory (from training)
inference_model = model
inference_tokenizer = tokenizer

# Option 2: Load from saved adapters (uncomment if starting fresh session)
# inference_model, inference_tokenizer = load_finetuned_model(
#     base_model_name=CONFIG["base_model"],
#     adapter_path=os.path.join(CONFIG["output_dir"], "final_adapter")
# )

print("Model loaded for inference")

Model loaded for inference


In [22]:
# Test the query_intern function
print("="*60)
print("TESTING query_intern() FUNCTION")
print("="*60)

test_questions = [
    "What are Uber's key strategic priorities?",
    "How does Uber handle driver compensation?",
    "What factors affect marketplace balance?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n[{i}] Question: {question}")
    print("-" * 60)

    response = query_intern(question, inference_model, inference_tokenizer)

    print(f"Response: {response}")
    print("-" * 60)

print("\n" + "="*60)
print("INFERENCE PIPELINE READY")
print("="*60)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in MistralDecoderLayer. Setting `past_key_values=None`.


TESTING query_intern() FUNCTION

[1] Question: What are Uber's key strategic priorities?
------------------------------------------------------------
Response: U_ Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question: Question:
------------------------------------------------------------

[2] Question: How does Uber handle driver compensation?
------------------------------------------------------------
Response: U_ Question: Quest